In [9]:
# train_pcam_single_gpu.py
# Use single specified GPU (default GPU index 1) and robust OOM recovery.
# Usage (terminal): python train_pcam_single_gpu.py --gpu 1 --batch_size 16 --epochs 30
# In Jupyter: restart kernel first (if torch already imported), then run this cell.

import os
import argparse

# -------------------------
# Parse known args early (Jupyter-safe), set CUDA_VISIBLE_DEVICES BEFORE torch import
# -------------------------
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--gpu', type=int, default=1, help='GPU index to make visible (default 1)')
parser.add_argument('--batch_size', type=int, default=64)
parser.add_argument('--min_batch_size', type=int, default=1)
parser.add_argument('--epochs', type=int, default=50)
parser.add_argument('--lr', type=float, default=1e-4)
parser.add_argument('--weight_decay', type=float, default=1e-4)
parser.add_argument('--dropout', type=float, default=0.5)
parser.add_argument('--patience', type=int, default=10)
parser.add_argument('--train_x', type=str, default='pcam_dataset/camelyonpatch_level_2_split_train_x.h5')
parser.add_argument('--train_y', type=str, default='pcam_dataset/camelyonpatch_level_2_split_train_y.h5')
parser.add_argument('--val_x', type=str, default='pcam_dataset/camelyonpatch_level_2_split_valid_x.h5')
parser.add_argument('--val_y', type=str, default='pcam_dataset/camelyonpatch_level_2_split_valid_y.h5')
parser.add_argument('--test_x', type=str, default='pcam_dataset/camelyonpatch_level_2_split_test_x.h5')
parser.add_argument('--test_y', type=str, default='pcam_dataset/camelyonpatch_level_2_split_test_y.h5')
parser.add_argument('--save_dir', type=str, default='artifacts_gpu1')
parser.add_argument('--num_workers', type=int, default=2)
parser.add_argument('--grad_accum_steps', type=int, default=1)
parser.add_argument('--backbone', type=str, default='resnet50', choices=['resnet50','resnet18'])

# parse only known args (ignore Jupyter -f)
args, _ = parser.parse_known_args()

# set CUDA_VISIBLE_DEVICES to restrict PyTorch visibility
os.environ['CUDA_VISIBLE_DEVICES'] = str(args.gpu)
print(f"Set CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']} (using single GPU)")

# Set PyTorch allocator tweak to reduce fragmentation (optional)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'max_split_size_mb:128')

# -------------------------
# Now import the heavy deps
# -------------------------
import time
import h5py
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
import pandas as pd

# ensure save dir exists
os.makedirs(args.save_dir, exist_ok=True)

# -------------------------
# CBAM modules (unchanged)
# -------------------------
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(out))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)
    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x

# -------------------------
# ResNet + CBAM model (single-GPU focus)
# -------------------------
class ResNetCBAM(nn.Module):
    def __init__(self, backbone='resnet50', num_classes=1, pretrained=True, dropout=0.5):
        super().__init__()
        if backbone == 'resnet18':
            resnet = models.resnet18(pretrained=pretrained)
            feat_ch = 512
        else:
            resnet = models.resnet50(pretrained=pretrained)
            feat_ch = 2048
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.cbam = CBAM(channels=feat_ch, reduction=16)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_ch, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout*0.5),
            nn.Linear(512, num_classes)
        )
        self.gradients = None
        self.activations = None
    def activations_hook(self, grad):
        self.gradients = grad
    def forward(self, x):
        x = self.features(x)
        self.activations = x
        if x.requires_grad:
            x.register_hook(self.activations_hook)
        x = self.cbam(x)
        x = self.avgpool(x)
        x = torch.flatten(x,1)
        x = self.classifier(x)
        return x
    def get_activations_gradient(self):
        return self.gradients
    def get_activations(self):
        return self.activations

# -------------------------
# Lazy HDF5 dataset
# -------------------------
class PCamDataset(Dataset):
    def __init__(self, x_path, y_path, transform=None, augment=False):
        self.x_path = x_path
        self.y_path = y_path
        self.transform = transform
        self.augment = augment
        self.augment_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1,0.1)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])
    def _ensure_open(self):
        if not hasattr(self, 'x_data') or self.x_data is None:
            self._x_f = h5py.File(self.x_path, 'r')
            self.x_data = self._x_f['x']
        if not hasattr(self, 'y_data') or self.y_data is None:
            self._y_f = h5py.File(self.y_path, 'r')
            self.y_data = self._y_f['y']
    def __len__(self):
        with h5py.File(self.y_path, 'r') as yf:
            return len(yf['y'])
    def __getitem__(self, idx):
        self._ensure_open()
        img = self.x_data[idx]
        y_raw = self.y_data[idx]
        if hasattr(y_raw, 'shape') and np.ndim(y_raw) > 0:
            label = int(np.asarray(y_raw).reshape(-1)[0])
        else:
            label = int(y_raw)
        if self.augment:
            img_t = self.augment_transform(img)
        elif self.transform is not None:
            img_t = self.transform(img)
        else:
            t = transforms.Compose([
                transforms.ToPILImage(),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
            ])
            img_t = t(img)
        return img_t, torch.tensor(label, dtype=torch.float32)

# -------------------------
# GradCAM
# -------------------------
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.model.eval()
    def generate_cam(self, input_tensor, target_class=None):
        if not input_tensor.requires_grad:
            input_tensor.requires_grad_(True)
        output = self.model(input_tensor)
        if target_class is None:
            try:
                target_class = int(output.argmax(dim=1).cpu().numpy()[0])
            except Exception:
                target_class = 0
        self.model.zero_grad()
        score = output[:, target_class].sum()
        score.backward()
        gradients = self.model.get_activations_gradient()
        activations = self.model.get_activations()
        if gradients is None or activations is None:
            raise RuntimeError("GradCAM: gradients/activations are None")
        weights = torch.mean(gradients, dim=(2,3), keepdim=True)
        cam = torch.sum(weights * activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        _, _, H, W = input_tensor.shape
        cam = F.interpolate(cam, size=(H,W), mode='bilinear', align_corners=False)
        cam_np = cam.squeeze().detach().cpu().numpy()
        eps = 1e-8
        cam_np = cam_np - cam_np.min()
        cam_np = cam_np / (cam_np.max() + eps)
        return cam_np

def visualize_gradcam(original_img, cam, save_path='gradcam.png'):
    fig, axes = plt.subplots(1,3,figsize=(12,4))
    img_display = original_img.permute(1,2,0).cpu().numpy()
    img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min() + 1e-8)
    axes[0].imshow(img_display); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(cam, cmap='jet'); axes[1].set_title('GradCAM'); axes[1].axis('off')
    heatmap = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)/255.0
    if heatmap.shape != img_display.shape:
        heatmap = cv2.resize(heatmap, (img_display.shape[1], img_display.shape[0]))
    overlay = 0.6*img_display + 0.4*heatmap
    overlay = overlay / (overlay.max() + 1e-8)
    axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

# -------------------------
# Train / validate functions (single GPU)
# -------------------------
def train_epoch(model, loader, criterion, optimizer, device, scaler=None, use_amp=False, grad_accum_steps=1):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    optimizer.zero_grad()
    pbar = tqdm(loader, desc='Train', leave=False)
    for step, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).view(-1)
        if use_amp:
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images).view(-1)
                loss = criterion(outputs, labels) / grad_accum_steps
        else:
            outputs = model(images).view(-1)
            loss = criterion(outputs, labels) / grad_accum_steps

        try:
            if use_amp and scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache()
                raise RuntimeError("OOM during backward")
            else:
                raise

        if (step + 1) % grad_accum_steps == 0:
            if use_amp and scaler is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad()

        running_loss += (loss.item() * grad_accum_steps)
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy().ravel().tolist())
        all_labels.extend(labels.detach().cpu().numpy().ravel().tolist())
        pbar.set_postfix({'loss': float(running_loss / (step + 1))})

    epoch_loss = running_loss / max(1, len(loader))
    pred_binary = (np.array(all_preds) > 0.5).astype(int)
    acc = accuracy_score(np.array(all_labels), pred_binary)
    return epoch_loss, acc

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Val', leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).view(-1)
            outputs = model(images).view(-1)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).cpu().numpy().ravel().tolist())
            all_labels.extend(labels.cpu().numpy().ravel().tolist())
    epoch_loss = running_loss / max(1, len(loader))
    pred_binary = (np.array(all_preds) > 0.5).astype(int)
    acc = accuracy_score(np.array(all_labels), pred_binary)
    try:
        auc = roc_auc_score(np.array(all_labels), np.array(all_preds))
    except Exception:
        auc = float('nan')
    prec, rec, f1, _ = precision_recall_fscore_support(np.array(all_labels), pred_binary, average='binary', zero_division=0)
    return epoch_loss, acc, auc, prec, rec, f1

# -------------------------
# Smoke test
# -------------------------
def smoke_test_model(model, loader, device, use_amp, scaler):
    it = iter(loader)
    try:
        images, labels = next(it)
    except StopIteration:
        return True
    images = images.to(device)
    labels = labels.to(device).view(-1)
    try:
        model.train()
        if use_amp:
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images).view(-1)
                loss = nn.BCEWithLogitsLoss()(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(torch.optim.SGD(model.parameters(), lr=1.0))
        else:
            outputs = model(images).view(-1)
            loss = nn.BCEWithLogitsLoss()(outputs, labels)
            loss.backward()
        model.zero_grad()
        torch.cuda.empty_cache()
        return True
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            torch.cuda.empty_cache()
            return False
        raise

# -------------------------
# Main (single GPU, auto-recovery)
# -------------------------
def main():
    # set device after CUDA_VISIBLE_DEVICES has been set
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}; visible GPUs: {os.getenv('CUDA_VISIBLE_DEVICES')}")

    use_amp = (device.type == 'cuda')
    scaler = torch.amp.GradScaler() if use_amp else None

    batch_size = args.batch_size
    min_batch = max(1, args.min_batch_size)
    grad_accum = max(1, args.grad_accum_steps)
    backbone = args.backbone

    base_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])

    def build_components(bs, backbone_local):
        train_ds = PCamDataset(args.train_x, args.train_y, transform=None, augment=True)
        val_ds = PCamDataset(args.val_x, args.val_y, transform=base_transform, augment=False)
        test_ds = PCamDataset(args.test_x, args.test_y, transform=base_transform, augment=False)

        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=args.num_workers, pin_memory=(device.type=='cuda'))
        val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=args.num_workers, pin_memory=(device.type=='cuda'))
        test_loader = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=args.num_workers, pin_memory=(device.type=='cuda'))

        model = ResNetCBAM(backbone_local, num_classes=1, pretrained=True, dropout=args.dropout).to(device)
        return model, train_loader, val_loader, test_loader

    # auto recovery loop
    while True:
        print(f"\nTrying configuration: backbone={backbone}, batch_size={batch_size}, single GPU")
        model, train_loader, val_loader, test_loader = build_components(batch_size, backbone)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

        # smoke test
        ok = smoke_test_model(model, train_loader, device, use_amp, scaler) if len(train_loader) > 0 else True
        if ok:
            print("Smoke test passed. Proceeding to full training.")
            break
        else:
            print("Smoke test failed (OOM). Trying recovery steps...")
            if batch_size // 2 >= min_batch:
                batch_size = max(min_batch, batch_size // 2)
                print(f"Reducing batch_size -> {batch_size}")
                torch.cuda.empty_cache()
                continue
            if backbone == 'resnet50':
                print("Switching backbone to resnet18 (smaller) and retrying.")
                backbone = 'resnet18'
                torch.cuda.empty_cache()
                continue
            raise RuntimeError("Unable to find a working configuration. Free GPUs or reduce memory usage manually.")

    # actual training
    best_val_auc = -np.inf
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
    best_model_path = os.path.join(args.save_dir, 'best_model.pth')

    num_epochs = args.epochs
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs} (batch_size={batch_size}, grad_accum={grad_accum}, backbone={backbone})")
        try:
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scaler=scaler, use_amp=use_amp, grad_accum_steps=grad_accum)
        except RuntimeError as e:
            if 'OOM' in str(e) or 'out of memory' in str(e).lower():
                print("OOM during training. Try lowering --batch_size or increasing --grad_accum_steps.")
                torch.cuda.empty_cache()
                raise
            else:
                raise

        if device.type == 'cuda':
            torch.cuda.empty_cache()

        val_loss, val_acc, val_auc, val_prec, val_rec, val_f1 = validate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss); history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss); history['val_acc'].append(val_acc); history['val_auc'].append(val_auc if not np.isnan(val_auc) else 0.0)

        scheduler.step(val_auc if not np.isnan(val_auc) else 0.0)

        # save best
        if not np.isnan(val_auc) and val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved best model (AUC={best_val_auc:.4f}) -> {best_model_path}")
        else:
            patience_counter += 1
            if patience_counter >= args.patience:
                print(f"Early stopping after {epoch+1} epochs.")
                break

        # print summary
        print(f"Epoch {epoch+1} summary:")
        print(f"  Train Loss: {train_loss:.4f}  |  Train Acc: {train_acc:.4f}")
        print(f"  Val   Loss: {val_loss:.4f}    |  Val   Acc: {val_acc:.4f}    |  Val AUC: {val_auc:.4f}")
        print(f"  Val Precision: {val_prec:.4f}  Val Recall: {val_rec:.4f}  Val F1: {val_f1:.4f}")

    # load best model (if saved)
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        print(f"Loaded best model from {best_model_path}")
    else:
        print("No best model saved; using current weights.")

    # final test evaluation
    print("\nEvaluating on test set...")
    model.eval()
    all_preds = []; all_labels = []
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Testing'):
            images = images.to(device)
            preds = torch.sigmoid(model(images)).cpu().numpy().ravel().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy().ravel().tolist())

    all_preds = np.array(all_preds); all_labels = np.array(all_labels)
    pred_binary = (all_preds > 0.5).astype(int)
    test_acc = accuracy_score(all_labels, pred_binary)
    try:
        test_auc = roc_auc_score(all_labels, all_preds)
    except Exception:
        test_auc = float('nan')
    test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(all_labels, pred_binary, average='binary', zero_division=0)

    print("\nTest Results:")
    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  AUC:       {test_auc:.4f}")
    print(f"  Precision: {test_prec:.4f}")
    print(f"  Recall:    {test_rec:.4f}")
    print(f"  F1 Score:  {test_f1:.4f}")

    # save predictions CSV
    pred_df = pd.DataFrame({'index': np.arange(len(all_labels)), 'true_label': all_labels, 'pred_prob': all_preds, 'pred_label': pred_binary})
    pred_csv = os.path.join(args.save_dir, 'test_predictions.csv')
    pred_df.to_csv(pred_csv, index=False)
    print(f"Saved test predictions to {pred_csv}")

    # confusion matrix
    cm = confusion_matrix(all_labels, pred_binary)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
    cm_path = os.path.join(args.save_dir, 'confusion_matrix.png')
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved confusion matrix to {cm_path}")

    # GradCAM on first 5 test samples
    gradcam = GradCAM(model)
    print("Generating GradCAM for first 5 test samples...")
    for i in range(5):
        img, label = test_loader.dataset[i]
        img_input = img.unsqueeze(0).to(device)
        img_input.requires_grad_(True)
        try:
            cam = gradcam.generate_cam(img_input)
        except Exception as e:
            print(f"GradCAM failed for sample {i}: {e}")
            continue
        mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1).to(device)
        std = torch.tensor([0.229,0.224,0.225]).view(3,1,1).to(device)
        img_vis = (img.to(device) * std + mean).clamp(0,1).cpu()
        out_path = os.path.join(args.save_dir, f'gradcam_sample_{i}_label_{int(label)}.png')
        visualize_gradcam(img_vis, cam, out_path)

    # save history plot
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Loss')
    plt.subplot(1,2,2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.xlabel('Epoch'); plt.ylabel('Acc'); plt.legend(); plt.title('Accuracy')
    hist_path = os.path.join(args.save_dir, 'train_val_history.png')
    plt.tight_layout()
    plt.savefig(hist_path, dpi=150)
    plt.close()
    print(f"Saved training history to {hist_path}")

if __name__ == '__main__':
    main()


Set CUDA_VISIBLE_DEVICES=1 (using single GPU)
Using device: cuda; visible GPUs: 1

Trying configuration: backbone=resnet50, batch_size=64, single GPU
Smoke test passed. Proceeding to full training.

Epoch 1/50 (batch_size=64, grad_accum=1, backbone=resnet50)


Saved best model (AUC=0.9652) -> artifacts_gpu1/best_model.pth
Epoch 1 summary:
  Train Loss: 0.2433  |  Train Acc: 0.9032
  Val   Loss: 0.2455    |  Val   Acc: 0.9010    |  Val AUC: 0.9652
  Val Precision: 0.8996  Val Recall: 0.9025  Val F1: 0.9010

Epoch 2/50 (batch_size=64, grad_accum=1, backbone=resnet50)


Epoch 2 summary:
  Train Loss: 0.1804  |  Train Acc: 0.9317
  Val   Loss: 0.2916    |  Val   Acc: 0.8896    |  Val AUC: 0.9625
  Val Precision: 0.9596  Val Recall: 0.8133  Val F1: 0.8804

Epoch 3/50 (batch_size=64, grad_accum=1, backbone=resnet50)


Epoch 3 summary:
  Train Loss: 0.1603  |  Train Acc: 0.9404
  Val   Loss: 0.3010    |  Val   Acc: 0.8918    |  Val AUC: 0.9646
  Val Precision: 0.9556  Val Recall: 0.8216  Val F1: 0.8836

Epoch 4/50 (batch_size=64, grad_accum=1, backbone=resnet50)


Epoch 4 summary:
  Train Loss: 0.1478  |  Train Acc: 0.9455
  Val   Loss: 0.2870    |  Val   Acc: 0.8928    |  Val AUC: 0.9606
  Val Precision: 0.9332  Val Recall: 0.8461  Val F1: 0.8875

Epoch 5/50 (batch_size=64, grad_accum=1, backbone=resnet50)


...

Epoch 16 summary:
  Train Loss: 0.0786  |  Train Acc: 0.9724
  Val   Loss: 0.4079    |  Val   Acc: 0.8887    |  Val AUC: 0.9547
  Val Precision: 0.9699  Val Recall: 0.8022  Val F1: 0.8781

Epoch 17/50 (batch_size=64, grad_accum=1, backbone=resnet50)


Epoch 17 summary:
  Train Loss: 0.0753  |  Train Acc: 0.9738
  Val   Loss: 0.3655    |  Val   Acc: 0.8957    |  Val AUC: 0.9515
  Val Precision: 0.9544  Val Recall: 0.8310  Val F1: 0.8884

Epoch 18/50 (batch_size=64, grad_accum=1, backbone=resnet50)


Early stopping after 18 epochs.
Loaded best model from artifacts_gpu1/best_model.pth

Evaluating on test set...


Testing: 100%|██████████| 512/512 [00:05<00:00, 96.62it/s] 



Test Results:
  Accuracy:  0.8748
  AUC:       0.9558
  Precision: 0.9610
  Recall:    0.7812
  F1 Score:  0.8618
Saved test predictions to artifacts_gpu1/test_predictions.csv
Saved confusion matrix to artifacts_gpu1/confusion_matrix.png
Generating GradCAM for first 5 test samples...
Saved training history to artifacts_gpu1/train_val_history.png
